# GBM hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [3]:
# Create subsets of the data for different training sizes - chronplogical order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="binary", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns identified for encoding:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


## Classification

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)

### 1k

In [5]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_1k_parallel.html")


[I 2026-04-22 15:44:25,353] A new study created in memory with name: no-name-8e730b5f-a715-4944-8443-aa3058edbc66
[I 2026-04-22 15:44:29,247] Trial 7 finished with value: 0.5 and parameters: {'n_estimators': 174, 'learning_rate': 0.04841237399008431, 'max_depth': 8, 'subsample': 0.3466109445898714, 'min_samples_split': 5, 'min_samples_leaf': 61, 'min_weight_fraction_leaf': 0.2572426118911079, 'min_impurity_decrease': 0.6617189568310914, 'max_features': 'sqrt', 'max_leaf_nodes': 25, 'ccp_alpha': 0.044952304550419285, 'imputer_type': 'knn', 'knn_neighbors': 14}. Best is trial 7 with value: 0.5.
[I 2026-04-22 15:44:29,739] Trial 1 finished with value: 0.5305944055944055 and parameters: {'n_estimators': 107, 'learning_rate': 0.030687405276438255, 'max_depth': 4, 'subsample': 0.4491485149575173, 'min_samples_split': 46, 'min_samples_leaf': 85, 'min_weight_fraction_leaf': 0.4754401813941242, 'min_impurity_decrease': 0.9672781052202073, 'max_features': 'sqrt', 'max_leaf_nodes': 24, 'ccp_alpha


BEST AUC: 0.6600
BEST PARAMETERS:
best_params = {
    "n_estimators": 468,
    "learning_rate": 0.009022964071123959,
    "max_depth": 6,
    "subsample": 0.7240087559596803,
    "min_samples_split": 28,
    "min_samples_leaf": 31,
    "min_weight_fraction_leaf": 0.009342492847849382,
    "min_impurity_decrease": 0.06849526577308163,
    "max_features": None,
    "max_leaf_nodes": 13,
    "ccp_alpha": 0.0047335814828048306,
    "imputer_type": "iterative",
}

--- PARAMETER IMPORTANCE ---
  min_samples_leaf    : 0.6802
  subsample           : 0.1734
  imputer_type        : 0.0509
  min_impurity_decrease: 0.0315
  min_weight_fraction_leaf: 0.0254
  max_depth           : 0.0115
  n_estimators        : 0.0092
  max_leaf_nodes      : 0.0055
  ccp_alpha           : 0.0041
  min_samples_split   : 0.0033
  max_features        : 0.0028
  learning_rate       : 0.0022


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_gbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0

imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print(f"Optuna Val AUC: {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")

Optuna Val AUC: 0.6600
Holdout Test AUC: 0.6920


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_10k_parallel.html")


[I 2026-04-22 15:49:45,949] A new study created in memory with name: no-name-005f701f-5b2a-4347-b803-7748f67740b4
[I 2026-04-22 15:50:05,035] Trial 3 finished with value: 0.5 and parameters: {'n_estimators': 437, 'learning_rate': 0.06889503112276328, 'max_depth': 7, 'subsample': 0.7266450332548319, 'min_samples_split': 9, 'min_samples_leaf': 68, 'min_weight_fraction_leaf': 0.4322677333928595, 'min_impurity_decrease': 0.16711738472740256, 'max_features': None, 'max_leaf_nodes': 6, 'ccp_alpha': 0.026589816822380555, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 3 with value: 0.5.
[I 2026-04-22 15:50:11,780] Trial 0 finished with value: 0.6811602075081582 and parameters: {'n_estimators': 903, 'learning_rate': 0.06252882049644057, 'max_depth': 5, 'subsample': 0.5485390708362865, 'min_samples_split': 40, 'min_samples_leaf': 39, 'min_weight_fraction_leaf': 0.4088955193448974, 'min_impurity_decrease': 0.37018697542358325, 'max_features': 'sqrt', 'max_leaf_nodes': 8, 'ccp


BEST AUC: 0.6970
BEST PARAMETERS:
best_params = {
    "n_estimators": 831,
    "learning_rate": 0.009193997709779543,
    "max_depth": 7,
    "subsample": 0.6525890903092806,
    "min_samples_split": 32,
    "min_samples_leaf": 26,
    "min_weight_fraction_leaf": 0.056461519342324906,
    "min_impurity_decrease": 0.11506112578571004,
    "max_features": "sqrt",
    "max_leaf_nodes": 10,
    "ccp_alpha": 1.5207831618216209e-05,
    "imputer_type": "simple",
    "simple_strategy": "mean",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.2034
  subsample           : 0.1552
  n_estimators        : 0.1430
  min_weight_fraction_leaf: 0.1332
  min_samples_split   : 0.0932
  max_leaf_nodes      : 0.0838
  learning_rate       : 0.0715
  min_samples_leaf    : 0.0511
  max_features        : 0.0328
  min_impurity_decrease: 0.0325
  imputer_type        : 0.0003
  max_depth           : 0.0000


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

Optuna Val AUC: 0.7030
Holdout Test AUC: 0.6888


### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_100k_parallel.html")


[I 2026-04-21 21:50:45,099] A new study created in memory with name: no-name-3fb7d702-7b14-4a89-8a89-1b7625dda2e1
[I 2026-04-21 21:50:50,161] Trial 5 finished with value: 0.6856218821328616 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 355, 'max_depth': 2, 'learning_rate': 0.0007531470236498383, 'scale_pos_weight': 8.062155351540042, 'min_split_gain': 12.600135939847666, 'min_child_weight': 0.027659028057181866, 'min_child_samples': 96, 'colsample_bytree': 0.4530169404337958, 'reg_alpha': 1.9476921791502694e-07, 'reg_lambda': 2.537413906629014e-06, 'colsample_bynode': 0.439201130695162, 'min_data_per_group': 231, 'max_cat_threshold': 882, 'cat_l2': 1.5992567061356557e-06, 'cat_smooth': 1.2492635955444809, 'max_cat_to_onehot': 41, 'max_bin': 363, 'subsample': 0.513304097853372, 'subsample_freq': 9, 'n_estimators': 3055}. Best is trial 5 with value: 0.6856218821328616.
[I 2026-04-21 21:51:22,869] Trial 6 finished with value: 0.702026953702141 and parameters: {'boosting_type': '


BEST AUC: 0.7139
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 291,
    "max_depth": 5,
    "learning_rate": 0.005951260639766576,
    "scale_pos_weight": 5.369026967967583,
    "min_split_gain": 0.5659664036067008,
    "min_child_weight": 8.234571214562124e-05,
    "min_child_samples": 344,
    "colsample_bytree": 0.3420503921726312,
    "reg_alpha": 8.6850113584894e-05,
    "reg_lambda": 0.05971126991196509,
    "colsample_bynode": 0.320861007484495,
    "min_data_per_group": 343,
    "max_cat_threshold": 285,
    "cat_l2": 0.11378821916587771,
    "cat_smooth": 0.050722191038187724,
    "max_cat_to_onehot": 11,
    "max_bin": 279,
    "top_rate": 0.1581569061472702,
    "other_rate": 0.45699552074780436,
    "n_estimators": 3930,
}

--- PARAMETER IMPORTANCE ---
  boosting_type       : 0.5198
  max_depth           : 0.0852
  min_split_gain      : 0.0780
  learning_rate       : 0.0763
  colsample_bytree    : 0.0576
  scale_pos_weight    : 0.0378
  mi

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

Optuna Val AUC: 0.7139
Holdout Test AUC: 0.7163


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_full_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Classification/optuna_gbm_full_parallel.html")


[I 2026-04-22 00:55:36,440] A new study created in memory with name: no-name-e6e86f3f-4af6-4c6c-bb30-bebed2ab97f3
[I 2026-04-22 00:56:27,391] Trial 3 finished with value: 0.7509030339384353 and parameters: {'boosting_type': 'goss', 'num_leaves': 377, 'max_depth': 2, 'learning_rate': 0.1903428179678173, 'scale_pos_weight': 6.082125486473243, 'min_split_gain': 26.39142477642419, 'min_child_weight': 4.8185524092618045e-05, 'min_child_samples': 290, 'colsample_bytree': 0.5066837006445346, 'reg_alpha': 0.008590451603378712, 'reg_lambda': 0.0006321751034820287, 'colsample_bynode': 0.6989172440719156, 'min_data_per_group': 976, 'max_cat_threshold': 632, 'cat_l2': 4.676533112163103e-06, 'cat_smooth': 34.70861426629406, 'max_cat_to_onehot': 37, 'max_bin': 433, 'top_rate': 0.7237115739628408, 'other_rate': 0.1719406111579936, 'n_estimators': 6050}. Best is trial 3 with value: 0.7509030339384353.
[I 2026-04-22 00:56:59,196] Trial 6 finished with value: 0.7442250170611076 and parameters: {'boostin


BEST AUC: 0.7566
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 298,
    "max_depth": 20,
    "learning_rate": 0.002488325776616227,
    "scale_pos_weight": 5.888271675991455,
    "min_split_gain": 1.6197064284930547,
    "min_child_weight": 0.0002644551677947709,
    "min_child_samples": 371,
    "colsample_bytree": 0.5316730321670442,
    "reg_alpha": 0.000426119399156468,
    "reg_lambda": 10.280323813334233,
    "colsample_bynode": 0.5281296236914715,
    "min_data_per_group": 996,
    "max_cat_threshold": 678,
    "cat_l2": 1.640465366517915e-08,
    "cat_smooth": 22.561055503199018,
    "max_cat_to_onehot": 45,
    "max_bin": 454,
    "top_rate": 0.18095067523535055,
    "other_rate": 0.31987875938170135,
    "n_estimators": 6725,
}

--- PARAMETER IMPORTANCE ---
  max_depth           : 0.2899
  min_data_per_group  : 0.1874
  max_bin             : 0.1858
  boosting_type       : 0.0868
  learning_rate       : 0.0573
  min_child_samples   : 0.0480
 

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingClassifier(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_gbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298


## Regression

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)

### 1k

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns identified for encoding:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_1k_parallel.html")


[I 2026-04-23 22:10:46,617] A new study created in memory with name: no-name-13f2af03-083b-43b5-8b65-32f439785209
[I 2026-04-23 22:10:51,413] Trial 1 finished with value: 0.3195284917601347 and parameters: {'n_estimators': 303, 'learning_rate': 0.006106045758624134, 'max_depth': 6, 'subsample': 0.6467541878256248, 'min_samples_split': 3, 'min_samples_leaf': 51, 'min_weight_fraction_leaf': 0.011498456804680812, 'min_impurity_decrease': 0.5846882487767013, 'max_features': 'sqrt', 'max_leaf_nodes': 25, 'ccp_alpha': 0.00024279340698643467, 'imputer_type': 'knn', 'knn_neighbors': 14}. Best is trial 1 with value: 0.3195284917601347.
[I 2026-04-23 22:10:53,791] Trial 6 finished with value: 0.32268776340152483 and parameters: {'n_estimators': 583, 'learning_rate': 0.18613907146227127, 'max_depth': 7, 'subsample': 0.3256554768663199, 'min_samples_split': 41, 'min_samples_leaf': 20, 'min_weight_fraction_leaf': 0.048746127324864, 'min_impurity_decrease': 0.8071696872387598, 'max_features': 'sqrt'


BEST RMSE: 0.3186
BEST PARAMETERS:
best_params = {
    "n_estimators": 740,
    "learning_rate": 0.015287664487358302,
    "max_depth": 3,
    "subsample": 0.7396849037377641,
    "min_samples_split": 24,
    "min_samples_leaf": 60,
    "min_weight_fraction_leaf": 0.301004593090821,
    "min_impurity_decrease": 0.3335701897157263,
    "max_features": "log2",
    "max_leaf_nodes": 13,
    "ccp_alpha": 7.01984244420758e-05,
    "imputer_type": "iterative",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.3137
  learning_rate       : 0.2791
  min_impurity_decrease: 0.0780
  min_weight_fraction_leaf: 0.0757
  min_samples_split   : 0.0661
  subsample           : 0.0464
  n_estimators        : 0.0384
  max_features        : 0.0298
  imputer_type        : 0.0239
  max_leaf_nodes      : 0.0208
  min_samples_leaf    : 0.0164
  max_depth           : 0.0117


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_gbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0

imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print(f"Optuna Val RMSE: {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")

Optuna Val RMSE: 0.3186
Holdout Test RMSE: 0.3348


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.3, 0.8), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 15)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_10k_parallel.html")


[I 2026-04-22 15:49:45,949] A new study created in memory with name: no-name-005f701f-5b2a-4347-b803-7748f67740b4
[I 2026-04-22 15:50:05,035] Trial 3 finished with value: 0.5 and parameters: {'n_estimators': 437, 'learning_rate': 0.06889503112276328, 'max_depth': 7, 'subsample': 0.7266450332548319, 'min_samples_split': 9, 'min_samples_leaf': 68, 'min_weight_fraction_leaf': 0.4322677333928595, 'min_impurity_decrease': 0.16711738472740256, 'max_features': None, 'max_leaf_nodes': 6, 'ccp_alpha': 0.026589816822380555, 'imputer_type': 'simple', 'simple_strategy': 'mean'}. Best is trial 3 with value: 0.5.
[I 2026-04-22 15:50:11,780] Trial 0 finished with value: 0.6811602075081582 and parameters: {'n_estimators': 903, 'learning_rate': 0.06252882049644057, 'max_depth': 5, 'subsample': 0.5485390708362865, 'min_samples_split': 40, 'min_samples_leaf': 39, 'min_weight_fraction_leaf': 0.4088955193448974, 'min_impurity_decrease': 0.37018697542358325, 'max_features': 'sqrt', 'max_leaf_nodes': 8, 'ccp


BEST AUC: 0.6970
BEST PARAMETERS:
best_params = {
    "n_estimators": 831,
    "learning_rate": 0.009193997709779543,
    "max_depth": 7,
    "subsample": 0.6525890903092806,
    "min_samples_split": 32,
    "min_samples_leaf": 26,
    "min_weight_fraction_leaf": 0.056461519342324906,
    "min_impurity_decrease": 0.11506112578571004,
    "max_features": "sqrt",
    "max_leaf_nodes": 10,
    "ccp_alpha": 1.5207831618216209e-05,
    "imputer_type": "simple",
    "simple_strategy": "mean",
}

--- PARAMETER IMPORTANCE ---
  ccp_alpha           : 0.2034
  subsample           : 0.1552
  n_estimators        : 0.1430
  min_weight_fraction_leaf: 0.1332
  min_samples_split   : 0.0932
  max_leaf_nodes      : 0.0838
  learning_rate       : 0.0715
  min_samples_leaf    : 0.0511
  max_features        : 0.0328
  min_impurity_decrease: 0.0325
  imputer_type        : 0.0003
  max_depth           : 0.0000


In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 2
best_params["learning_rate"] = original_lr / 2

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

Optuna Val AUC: 0.7030
Holdout Test AUC: 0.6888


### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_100k_parallel.html")


[I 2026-04-21 21:50:45,099] A new study created in memory with name: no-name-3fb7d702-7b14-4a89-8a89-1b7625dda2e1
[I 2026-04-21 21:50:50,161] Trial 5 finished with value: 0.6856218821328616 and parameters: {'boosting_type': 'gbdt', 'num_leaves': 355, 'max_depth': 2, 'learning_rate': 0.0007531470236498383, 'scale_pos_weight': 8.062155351540042, 'min_split_gain': 12.600135939847666, 'min_child_weight': 0.027659028057181866, 'min_child_samples': 96, 'colsample_bytree': 0.4530169404337958, 'reg_alpha': 1.9476921791502694e-07, 'reg_lambda': 2.537413906629014e-06, 'colsample_bynode': 0.439201130695162, 'min_data_per_group': 231, 'max_cat_threshold': 882, 'cat_l2': 1.5992567061356557e-06, 'cat_smooth': 1.2492635955444809, 'max_cat_to_onehot': 41, 'max_bin': 363, 'subsample': 0.513304097853372, 'subsample_freq': 9, 'n_estimators': 3055}. Best is trial 5 with value: 0.6856218821328616.
[I 2026-04-21 21:51:22,869] Trial 6 finished with value: 0.702026953702141 and parameters: {'boosting_type': '


BEST AUC: 0.7139
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 291,
    "max_depth": 5,
    "learning_rate": 0.005951260639766576,
    "scale_pos_weight": 5.369026967967583,
    "min_split_gain": 0.5659664036067008,
    "min_child_weight": 8.234571214562124e-05,
    "min_child_samples": 344,
    "colsample_bytree": 0.3420503921726312,
    "reg_alpha": 8.6850113584894e-05,
    "reg_lambda": 0.05971126991196509,
    "colsample_bynode": 0.320861007484495,
    "min_data_per_group": 343,
    "max_cat_threshold": 285,
    "cat_l2": 0.11378821916587771,
    "cat_smooth": 0.050722191038187724,
    "max_cat_to_onehot": 11,
    "max_bin": 279,
    "top_rate": 0.1581569061472702,
    "other_rate": 0.45699552074780436,
    "n_estimators": 3930,
}

--- PARAMETER IMPORTANCE ---
  boosting_type       : 0.5198
  max_depth           : 0.0852
  min_split_gain      : 0.0780
  learning_rate       : 0.0763
  colsample_bytree    : 0.0576
  scale_pos_weight    : 0.0378
  mi

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")
    
# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

Optuna Val AUC: 0.7139
Holdout Test AUC: 0.7163


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_gbm(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "subsample": trial.suggest_float("subsample", 0.1, 1.0), # Fraction of samples used for training each tree
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 500), # Min samples required to split a node
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "min_weight_fraction_leaf": trial.suggest_float("min_weight_fraction_leaf", 0, 0.5), # Min weight fraction required in a leaf node
        "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0, 1.0), # Min impurity decrease required to make a split
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "ccp_alpha": trial.suggest_float("ccp_alpha", 1e-5, 0.1, log=True), # Complexity parameter for pruning
        "random_state": 42, # Fixed random state for reproducibility
        "verbose": 0, # Set to 0 to disable verbose output during training
    }

    imputer_type = trial.suggest_categorical("imputer_type", ["simple", "knn", "iterative"])
    
    if imputer_type == "simple":
        strategy = trial.suggest_categorical("simple_strategy", ["mean", "median", "most_frequent"])
        imputer = SimpleImputer(strategy=strategy)
        
    elif imputer_type == "knn": # K Nearest Neighbors Imputer
        n_neighbors = trial.suggest_int("knn_neighbors", 3, 50)
        imputer = KNNImputer(n_neighbors=n_neighbors)
        
    else: # iterative
        imputer = IterativeImputer(max_iter=10, random_state=42)

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        imputer,
        GradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_gbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_gbm.optimize(objective_gbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_gbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_gbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_gbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_gbm)
fig1.show()
    
fig2 = plot_param_importances(study_gbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(
    study_gbm,
    params=[
        "n_estimators", "learning_rate", "max_depth", "subsample", 
        "min_samples_split", "min_samples_leaf", "min_weight_fraction_leaf",
        "min_impurity_decrease", "max_features", "max_leaf_nodes",
        "ccp_alpha", "imputer_type"
    ]
)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_full_history.html")
fig2.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/GBM/Regression/optuna_gbm_full_parallel.html")


[I 2026-04-22 00:55:36,440] A new study created in memory with name: no-name-e6e86f3f-4af6-4c6c-bb30-bebed2ab97f3
[I 2026-04-22 00:56:27,391] Trial 3 finished with value: 0.7509030339384353 and parameters: {'boosting_type': 'goss', 'num_leaves': 377, 'max_depth': 2, 'learning_rate': 0.1903428179678173, 'scale_pos_weight': 6.082125486473243, 'min_split_gain': 26.39142477642419, 'min_child_weight': 4.8185524092618045e-05, 'min_child_samples': 290, 'colsample_bytree': 0.5066837006445346, 'reg_alpha': 0.008590451603378712, 'reg_lambda': 0.0006321751034820287, 'colsample_bynode': 0.6989172440719156, 'min_data_per_group': 976, 'max_cat_threshold': 632, 'cat_l2': 4.676533112163103e-06, 'cat_smooth': 34.70861426629406, 'max_cat_to_onehot': 37, 'max_bin': 433, 'top_rate': 0.7237115739628408, 'other_rate': 0.1719406111579936, 'n_estimators': 6050}. Best is trial 3 with value: 0.7509030339384353.
[I 2026-04-22 00:56:59,196] Trial 6 finished with value: 0.7442250170611076 and parameters: {'boostin


BEST AUC: 0.7566
BEST PARAMETERS:
best_params = {
    "boosting_type": "goss",
    "num_leaves": 298,
    "max_depth": 20,
    "learning_rate": 0.002488325776616227,
    "scale_pos_weight": 5.888271675991455,
    "min_split_gain": 1.6197064284930547,
    "min_child_weight": 0.0002644551677947709,
    "min_child_samples": 371,
    "colsample_bytree": 0.5316730321670442,
    "reg_alpha": 0.000426119399156468,
    "reg_lambda": 10.280323813334233,
    "colsample_bynode": 0.5281296236914715,
    "min_data_per_group": 996,
    "max_cat_threshold": 678,
    "cat_l2": 1.640465366517915e-08,
    "cat_smooth": 22.561055503199018,
    "max_cat_to_onehot": 45,
    "max_bin": 454,
    "top_rate": 0.18095067523535055,
    "other_rate": 0.31987875938170135,
    "n_estimators": 6725,
}

--- PARAMETER IMPORTANCE ---
  max_depth           : 0.2899
  min_data_per_group  : 0.1874
  max_bin             : 0.1858
  boosting_type       : 0.0868
  learning_rate       : 0.0573
  min_child_samples   : 0.0480
 

In [ ]:
# Check overfit on holdout set with best parameters from tuning

# Create a copy to avoid modifying Optuna's internal dictionary
best_params = study_gbm.best_params.copy()

# Extract imputer settings safely before applying scaling trick
imputer_type = best_params.pop("imputer_type")

if imputer_type == "simple":
    strategy = best_params.pop("simple_strategy")
    best_imputer = SimpleImputer(strategy=strategy)
    print(f"SimpleImputer strategy: {strategy}")
elif imputer_type == "knn":
    n_neighbors = best_params.pop("knn_neighbors")
    best_imputer = KNNImputer(n_neighbors=n_neighbors)
    print(f"KNNImputer n_neighbors: {n_neighbors}")
else:
    best_imputer = IterativeImputer(max_iter=10, random_state=42)
    print("IterativeImputer selected")

# Clean up parameter dictionary
best_params.pop("simple_strategy", None)
best_params.pop("knn_neighbors", None)

# Scaling trick
original_trees = best_params["n_estimators"]
original_lr = best_params["learning_rate"]

best_params["n_estimators"] = original_trees * 10
best_params["learning_rate"] = original_lr / 10

print(f"\n[Scaling Trick Applied] SKLEARN GBM: Trees {original_trees} -> {best_params['n_estimators']}, LR {original_lr:.4f} -> {best_params['learning_rate']:.4f}")

# Add fixed parameters
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    best_imputer,
    GradientBoostingRegressor(**best_params)
)

# Fit on the entire tuning set
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_gbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298
